In [ ]:
import pandas as pd
import numpy as np

# ===== 1️⃣ Read Data =====
df = pd.read_csv("/root/autodl-tmp/MELD/meld_test_prosody.csv")

# ===== 2️⃣ Handle Missing F0 =====
df.loc[df["voiced_ratio"] == 0, ["f0_std_hz", "f0_slope_hz_per_sec"]] = np.nan

# ===== 3️⃣ Quantile Binning Function =====
def quantile_bin(series):
    q1 = series.quantile(0.33)
    q2 = series.quantile(0.66)

    def label(x):
        if pd.isna(x):
            return "missing"
        elif x <= q1:
            return "low"
        elif x <= q2:
            return "mid"
        else:
            return "high"

    return series.apply(label)

# ===== 4️⃣ Discretize Core Features =====
df["loudness_bin"] = quantile_bin(df["rms_mean"])
df["loudness_dynamics_bin"] = quantile_bin(df["rms_range_p95_p05"])
df["pitch_variability_bin"] = quantile_bin(df["f0_std_hz"])
df["voicing_bin"] = quantile_bin(df["voiced_ratio"])
df["rate_bin"] = quantile_bin(df["rate_proxy_peaks_per_sec"])

# ===== 5️⃣ Process Slope Separately =====
def slope_bin(x):
    if pd.isna(x):
        return "missing"
    elif x < -10:
        return "falling"
    elif x > 10:
        return "rising"
    else:
        return "flat"

df["pitch_trend"] = df["f0_slope_hz_per_sec"].apply(slope_bin)

# ===== 6️⃣ Simple Duration Discretization (Optional) =====
dur_q1 = df["duration_sec"].quantile(0.33)
dur_q2 = df["duration_sec"].quantile(0.66)

def duration_bin(x):
    if x <= dur_q1:
        return "short"
    elif x <= dur_q2:
        return "mid"
    else:
        return "long"

df["duration_bin"] = df["duration_sec"].apply(duration_bin)

# ===== 7️⃣ Print Distribution for Each Bin (for Checking) =====
cols_to_check = [
    "loudness_bin",
    "loudness_dynamics_bin",
    "pitch_variability_bin",
    "voicing_bin",
    "rate_bin",
    "pitch_trend",
    "duration_bin"
]

for col in cols_to_check:
    print(f"\n=== {col} ===")
    print(df[col].value_counts())

# ===== 8️⃣ Save New File =====
df.to_csv("test_audio_features_binned.csv", index=False)

print("\n✅ Discretization completed, file saved as audio_features_binned.csv")